This experiment is dedicated to test $\hat R_{\nu}$ on different bivariate normal distribution with diagonal covariance.

In [ ]:
!pip uninstall -yq jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt tensorflow-probability
# heads out since jax might drop support for cuda 12, currently (as of Aug 10, 2026), JAX has issues with CUDA 13
# see this post: https://github.com/jax-ml/jax/issues/37923
!pip install -Uq "jax[cuda12]" tfp-nightly blackjax inference_gym optax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 133.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.9/390.9 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 145.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.2/188.2 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 MB 30.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires tensorflow-probability>=0.13.0, which is not installed.


### **Package Import and Other Setups**

In [ ]:
# GPU set up to accelerate performance
import os
# in case jax eats up my GPU RAM
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ['XLA_FLAGS'] = (
    '--xla_gpu_triton_gemm_any=True '
    '--xla_gpu_enable_latency_hiding_scheduler=true '
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from jax import random, jit, vmap, lax
import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions
import inference_gym.using_jax as gym
import jaxlib
import blackjax

import optax

# import arviz as az
# import arviz_stats as avs

import warnings
warnings.filterwarnings('ignore')

import psutil

import gc

from google.colab import drive
from matplotlib.lines import Line2D

process = psutil.Process(os.getpid())

def mem(msg):
    print(f"{msg}: {process.memory_info().rss / 1024**2:.1f} MB")

# verification to make sure this is on a GPU
print(jax.devices())
print(jax.default_backend())

[CudaDevice(id=0)]
gpu


In [ ]:
drive.mount('/content/drive')
utility_link = '/content/drive/MyDrive/JHU Stuff/Capstone/Utility_Functions/BJX_Utility.py'
with open(utility_link) as f: exec(f.read())

Mounted at /content/drive


### **Hyperparam Setups**

In [ ]:
max_warmup = 1000
warmup_window = 100

window_array = np.append(np.repeat(10, 10),
                      np.repeat(warmup_window, max_warmup // warmup_window - 1))

warmup_length = np.repeat(10, len(window_array))
for i in range(len(warmup_length) - 1):
    warmup_length[i + 1] = warmup_length[i] + window_array[i + 1]

# Transition kernel for short regime
repitition = 10
num_chains_short = 2048
num_super_chains = 16

In [ ]:
# quantiles for chi squared with df = 1
chi_up = 3.841459 # 95th quantile for chi squared with df = 1
chi_lo = 0.00393214  # 05th quantile for chi squared with df = 1
tau = 1e-4
M = num_chains_short // num_super_chains
nRhat_lower = np.sqrt(1 + 1 / M + tau)
eps_lower = nRhat_lower - 1
bound = [chi_lo / num_chains_short, chi_up / num_chains_short]
threshold = eps_lower

In [ ]:
def bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir):
  # mean array
  mu = jnp.full(num_dim, 0.0)
  # covariance matrix
  cov = jnp.array([
    [var1, rho*jnp.sqrt(var1*var2)],
    [rho*jnp.sqrt(var1*var2), var2]])
  target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=cov)
  init_step_size = 0.5
  def target_log_prob_fn(x):
    return target.log_prob(x)
  def initialize(shape, key):
    offset = 2
    return 10*random.normal(key, shape + (num_dim,)) + offset
  mean_benchmark = target.mean()
  var_benchmark = target.variance()

  #simulation part:
  Aniso_MSE_c_list = []
  Aniso_MSE_n_list = []
  Aniso_RHat_c_list = []
  Aniso_RHat_n_list = []

  base_key = random.PRNGKey(0)
  keys = random.split(base_key, repitition)
  for length in warmup_length:
    mem(f"Simulation Start")
    simulation(length,num_chains_short, num_super_chains,
               False,initialize, keys, target_log_prob_fn,init_step_size,
               repitition, Aniso_RHat_c_list,Aniso_MSE_c_list,mean_benchmark,var_benchmark)
    simulation(length,num_chains_short, num_super_chains,
               True,initialize, keys, target_log_prob_fn,init_step_size,
               repitition, Aniso_RHat_n_list,Aniso_MSE_n_list, mean_benchmark,var_benchmark)
  # save to dataframe and files
  MSE_c_df = pd.DataFrame(Aniso_MSE_c_list)
  R_Hat_c_df = pd.DataFrame(Aniso_RHat_c_list)
  MSE_n_df = pd.DataFrame(Aniso_MSE_n_list)
  R_Hat_n_df = pd.DataFrame(Aniso_RHat_n_list)

  # Use a descriptive parameter label
  label = f"var1_{var1:g}_var2_{var2:g}_rho_{rho:g}"

  MSE_c_df.to_pickle(f"{save_dir}/{label}_MSE_c.pkl")
  MSE_n_df.to_pickle(f"{save_dir}/{label}_MSE_n.pkl")
  R_Hat_c_df.to_pickle(f"{save_dir}/{label}_Rhat_c.pkl")
  R_Hat_n_df.to_pickle(f"{save_dir}/{label}_Rhat_n.pkl")

  del MSE_c_df, R_Hat_c_df, MSE_n_df, R_Hat_n_df
  gc.collect()
  print("Simulation Done, Data Saved!")


### **Experiment Part**

In [ ]:
# Set up File Directory
save_dir = "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Bi_Equal/"

### **Isotropic Diagonal Bivariate MVN**

#### **Distribution with covariance matrix:**
$$
\begin{bmatrix}
  1 & \rho \\
  \rho & 1 \\
\end{bmatrix}
$$

In [ ]:
var1 = 1.0
var2 = 1.0
num_dim = 2

In [ ]:
rho = 0.0
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1518.8 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0006598933832719922
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0006267476128414273
Simulation Start: 1897.2 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0006081467727199197
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0005494750221259892
Simulation Start: 1908.0 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0005718282191082835
Naive initialization. Warmup Length: 30; mean of MSE is: 0.00040034629637375474
Simulation Start: 1915.3 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0005899652605876327
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0003897018905263394
Simulation Start: 1917.9 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0003614262386690825
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0004703286977019161
Simulation Start: 1918.7 MB
Constrained

In [ ]:
rho = 0.3
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1932.4 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0003945356293115765
Naive initialization. Warmup Length: 10; mean of MSE is: 0.00031876866705715656
Simulation Start: 1937.6 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0005520757986232638
Naive initialization. Warmup Length: 20; mean of MSE is: 0.000569471507333219
Simulation Start: 1938.1 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0007268766057677567
Naive initialization. Warmup Length: 30; mean of MSE is: 0.000616035598795861
Simulation Start: 1938.5 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0005833248142153025
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0007237005629576743
Simulation Start: 1938.7 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.00031896348809823394
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0003432332305237651
Simulation Start: 1938.8 MB
Constrained 

In [ ]:
rho = 0.6
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1941.2 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.007753681391477585
Naive initialization. Warmup Length: 10; mean of MSE is: 0.00015761285612825304
Simulation Start: 1941.3 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.000547951553016901
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0006078197038732469
Simulation Start: 1941.5 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0008792093140073121
Naive initialization. Warmup Length: 30; mean of MSE is: 0.000809245917480439
Simulation Start: 1941.5 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0004922159714624286
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0005346411489881575
Simulation Start: 1941.6 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0003807678585872054
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0003663685929495841
Simulation Start: 1941.7 MB
Constrained in

In [ ]:
rho = 0.9
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1942.9 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.021145159378647804
Naive initialization. Warmup Length: 10; mean of MSE is: 0.13955560326576233
Simulation Start: 1945.5 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0008726661326363683
Naive initialization. Warmup Length: 20; mean of MSE is: 0.01120464876294136
Simulation Start: 1945.6 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0008657045545987785
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0010428240057080984
Simulation Start: 1945.7 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0006058949511498213
Naive initialization. Warmup Length: 40; mean of MSE is: 0.000698956660926342
Simulation Start: 1945.7 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0004345464112702757
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0007046094397082925
Simulation Start: 1945.8 MB
Constrained initia

In [ ]:
rho = -0.3
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1946.5 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0004910908755846322
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0006028377683833241
Simulation Start: 1946.5 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.00044725072802975774
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0003466297057457268
Simulation Start: 1946.5 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.00038377600139938295
Naive initialization. Warmup Length: 30; mean of MSE is: 0.000529159267898649
Simulation Start: 1946.5 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0007439890177920461
Naive initialization. Warmup Length: 40; mean of MSE is: 0.00027223321376368403
Simulation Start: 1946.6 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.00046328239841386676
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00040408052154816687
Simulation Start: 1946.6 MB
Constrai

In [ ]:
rho = -0.6
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1947.4 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.006287361029535532
Naive initialization. Warmup Length: 10; mean of MSE is: 0.010772819630801678
Simulation Start: 1949.7 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.00041854954906739295
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0005139147397130728
Simulation Start: 1949.7 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0004660918493755162
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0005062940763309598
Simulation Start: 1949.7 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0002917599049396813
Naive initialization. Warmup Length: 40; mean of MSE is: 0.00033892429200932384
Simulation Start: 1949.8 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0006096415454521775
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0005975562380626798
Simulation Start: 1949.8 MB
Constrained 

In [ ]:
rho = -0.9
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1950.5 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0005710297846235335
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0005043837591074407
Simulation Start: 1950.6 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0004297954437788576
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00014565711899194866
Simulation Start: 1950.6 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0003924463817384094
Naive initialization. Warmup Length: 30; mean of MSE is: 0.00035596726229414344
Simulation Start: 1950.7 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0005267067463137209
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0003909518418367952
Simulation Start: 1950.8 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.00032549985917285085
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0004402264894451946
Simulation Start: 1951.1 MB
Constrain

#### **Distribution with covariance matrix:**
$$
\begin{bmatrix}
  10 & \rho*10 \\
  \rho*10 & 10 \\
\end{bmatrix}
$$

In [ ]:
var1 = 10.0
var2 = 10.0
num_dim = 2

In [ ]:
rho = 0.0
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1951.6 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0004632581549230963
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0004980959929525852
Simulation Start: 1952.8 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0003676768974401057
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00039009033935144544
Simulation Start: 1952.8 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0005968016921542585
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0006402850849553943
Simulation Start: 1952.8 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0004089563153684139
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0003199530183337629
Simulation Start: 1952.8 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0006014524260535836
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0005111273494549096
Simulation Start: 1952.8 MB
Constrained

In [ ]:
rho = 0.3
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1953.3 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0006524756317958236
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0005501275882124901
Simulation Start: 1953.3 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0003270870656706393
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0004122150712646544
Simulation Start: 1953.3 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0005912893102504313
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0004154485650360584
Simulation Start: 1953.3 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.00038647817564196885
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0004728646599687636
Simulation Start: 1953.3 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0004269477794878185
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0005127767217345536
Simulation Start: 1953.3 MB
Constrained

In [ ]:
rho = 0.6
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1953.8 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.004895365796983242
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0022922074422240257
Simulation Start: 1953.8 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0004081553779542446
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0002555932733230293
Simulation Start: 1953.8 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0005528643960133195
Naive initialization. Warmup Length: 30; mean of MSE is: 0.00046098517486825585
Simulation Start: 1953.8 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.00057172030210495
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0005567671032622457
Simulation Start: 1953.8 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0002935614320449531
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00033178305602632463
Simulation Start: 1953.8 MB
Constrained i

In [ ]:
rho = 0.9
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1954.3 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.06285028904676437
Naive initialization. Warmup Length: 10; mean of MSE is: 0.031972240656614304
Simulation Start: 1954.3 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.004995841532945633
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0024810165632516146
Simulation Start: 1954.3 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.000682560377754271
Naive initialization. Warmup Length: 30; mean of MSE is: 0.00043247820576652884
Simulation Start: 1954.4 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0007847541128285229
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0003978954628109932
Simulation Start: 1954.4 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.000455579865956679
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00045667338417842984
Simulation Start: 1954.4 MB
Constrained init

In [ ]:
rho = -0.3
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1954.9 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0005738228792324662
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0005670354003086686
Simulation Start: 1954.9 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0004332336247898638
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0003819313715212047
Simulation Start: 1954.9 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0004454449808690697
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0005371019942685962
Simulation Start: 1954.9 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.000492289022076875
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0005286045488901436
Simulation Start: 1954.9 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.00042934916564263403
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0004827052471227944
Simulation Start: 1954.9 MB
Constrained 

In [ ]:
rho = -0.6
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1955.1 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.00195838650688529
Naive initialization. Warmup Length: 10; mean of MSE is: 0.00046166699030436575
Simulation Start: 1955.2 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.00028340250719338655
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00029883309616707265
Simulation Start: 1955.2 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0005260318866930902
Naive initialization. Warmup Length: 30; mean of MSE is: 0.00046268152073025703
Simulation Start: 1955.2 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.00048252861597575247
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0005618768045678735
Simulation Start: 1955.2 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0006099893362261355
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0005352656007744372
Simulation Start: 1955.2 MB
Constrain

In [ ]:
rho = -0.9
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1955.3 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.02620440162718296
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0008698858437128365
Simulation Start: 1955.5 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.002227721270173788
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0001601386466063559
Simulation Start: 1955.5 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0002700606710277498
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0004223128198646009
Simulation Start: 1955.5 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0008287766249850392
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0007924053934402764
Simulation Start: 1955.5 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0007276671240106225
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0007131014717742801
Simulation Start: 1955.5 MB
Constrained ini

#### 3. Distribution with covariance matrix:
$$
\begin{bmatrix}
  100 & \rho*100 \\
  \rho*100 & 100 \\
\end{bmatrix}
$$

In [ ]:
var1 = 100.0
var2 = 100.0
num_dim = 2

In [ ]:
rho = 0.0
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1520.9 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0005377880879677832
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0005579747958108783
Simulation Start: 1894.8 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.000545137154404074
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00036784267285838723
Simulation Start: 1908.3 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0005825413973070681
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0005772901931777596
Simulation Start: 1916.3 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0004335752746555954
Naive initialization. Warmup Length: 40; mean of MSE is: 0.00048745059757493436
Simulation Start: 1919.6 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0004339453880675137
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00048758351476863027
Simulation Start: 1920.9 MB
Constraine

In [ ]:
rho = 0.3
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1934.0 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0007651732885278761
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0005182693130336702
Simulation Start: 1939.4 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.00031380687141790986
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0003964726347476244
Simulation Start: 1939.9 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0004574542981572449
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0004877737374044955
Simulation Start: 1940.0 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0005530972266569734
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0004506880068220198
Simulation Start: 1940.1 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0003560819895938039
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0004169860912952572
Simulation Start: 1940.1 MB
Constrained

In [ ]:
rho = 0.6
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1944.0 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.003966915886849165
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0025663156993687153
Simulation Start: 1944.3 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0003581687924452126
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00034159532515332103
Simulation Start: 1944.5 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0005165135371498764
Naive initialization. Warmup Length: 30; mean of MSE is: 0.00045502945431508124
Simulation Start: 1944.5 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0005258667515590787
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0005333733861334622
Simulation Start: 1944.6 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.00035823279176838696
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0002139421849278733
Simulation Start: 1944.7 MB
Constraine

In [ ]:
rho = 0.9
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1945.9 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.042306818068027496
Naive initialization. Warmup Length: 10; mean of MSE is: 0.02494368515908718
Simulation Start: 1949.2 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.02515042945742607
Naive initialization. Warmup Length: 20; mean of MSE is: 0.01266102958470583
Simulation Start: 1949.2 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.010443122126162052
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0050948988646268845
Simulation Start: 1949.2 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.006281075067818165
Naive initialization. Warmup Length: 40; mean of MSE is: 0.002407032297924161
Simulation Start: 1949.2 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0020308741368353367
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0007909525884315372
Simulation Start: 1949.2 MB
Constrained initializa

In [ ]:
rho = -0.3
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1950.4 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0005025292048230767
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0005405372357927263
Simulation Start: 1950.5 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.00044639199040830135
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00046862097224220634
Simulation Start: 1950.6 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0005933476495556533
Naive initialization. Warmup Length: 30; mean of MSE is: 0.00038076331838965416
Simulation Start: 1950.7 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0004452928842511028
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0005680119502358139
Simulation Start: 1950.7 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0004870750126428902
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0006612600991502404
Simulation Start: 1950.7 MB
Constrain

In [ ]:
rho = -0.6
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1951.1 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0018606142839416862
Naive initialization. Warmup Length: 10; mean of MSE is: 0.000377584045054391
Simulation Start: 1952.7 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0002854185877367854
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0002445676364004612
Simulation Start: 1952.7 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.00045056449016556144
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0005364162498153746
Simulation Start: 1952.7 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0007838690653443336
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0008931603515520692
Simulation Start: 1953.0 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0005917154485359788
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0005536582320928574
Simulation Start: 1953.2 MB
Constrained 

In [ ]:
rho = -0.9
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1953.5 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.02119620516896248
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0004587688890751451
Simulation Start: 1953.5 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.014454755000770092
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00026656343834474683
Simulation Start: 1953.5 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.004774430301040411
Naive initialization. Warmup Length: 30; mean of MSE is: 0.00041585424332879484
Simulation Start: 1953.6 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.003593429923057556
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0008200680022127926
Simulation Start: 1953.6 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.002071414841338992
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0008724119397811592
Simulation Start: 1953.6 MB
Constrained init

#### 4. Distribution with covariance matrix:
$$
\begin{bmatrix}
  1000 & \rho*1000 \\
  \rho*1000 & 1000 \\
\end{bmatrix}
$$

In [ ]:
var1 = 1000.0
var2 = 1000.0
num_dim = 2

In [ ]:
rho = 0.0
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1953.9 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0005398689536377788
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0005089539336040616
Simulation Start: 1954.9 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.00031951419077813625
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00043323534191586077
Simulation Start: 1954.9 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0006681645172648132
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0007170335156843066
Simulation Start: 1954.9 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0004574654740281403
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0006232906016521156
Simulation Start: 1954.9 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0005104176234453917
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0006170413689687848
Simulation Start: 1954.9 MB
Constraine

In [ ]:
rho = 0.3
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1955.3 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0007773374672979116
Naive initialization. Warmup Length: 10; mean of MSE is: 0.000645400257781148
Simulation Start: 1955.3 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0004174253554083407
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00042328290874138474
Simulation Start: 1955.3 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.00046099285827949643
Naive initialization. Warmup Length: 30; mean of MSE is: 0.00038131180917844176
Simulation Start: 1955.3 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0005371238803490996
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0004925599205307662
Simulation Start: 1955.3 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.00019762851297855377
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0004218677640892565
Simulation Start: 1955.3 MB
Constrain

In [ ]:
rho = 0.6
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1955.5 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0018463203450664878
Naive initialization. Warmup Length: 10; mean of MSE is: 0.001224546111188829
Simulation Start: 1955.8 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0004015562008135021
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0003096907748840749
Simulation Start: 1955.9 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0004982095560990274
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0004931146395392716
Simulation Start: 1955.9 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0004130784946028143
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0004970406298525631
Simulation Start: 1955.9 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.00029669090872630477
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0003458958235569298
Simulation Start: 1955.9 MB
Constrained 

In [ ]:
rho = 0.9
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1956.0 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.005219853483140469
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0030130057130008936
Simulation Start: 1956.0 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.002566944109275937
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0016768418718129396
Simulation Start: 1956.1 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0016961790388450027
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0007842688937671483
Simulation Start: 1956.1 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0011184306349605322
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0007394651765935123
Simulation Start: 1956.1 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0006690224399790168
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0006336225778795779
Simulation Start: 1956.1 MB
Constrained in

In [ ]:
rho = -0.3
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1956.4 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0006296056089922786
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0005998363485559821
Simulation Start: 1956.4 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.00035762731567956507
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00022161468223202974
Simulation Start: 1956.4 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0005332378786988556
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0005410852027125657
Simulation Start: 1956.4 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0005075237713754177
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0007278042612597346
Simulation Start: 1956.4 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.00059012311976403
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0005537833203561604
Simulation Start: 1956.4 MB
Constrained 

In [ ]:
rho = -0.6
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1956.5 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0009614286827854812
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0005802514497190714
Simulation Start: 1956.7 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0003025599871762097
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0002454761415719986
Simulation Start: 1956.7 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.000490140519104898
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0005671658436767757
Simulation Start: 1956.7 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0007058033952489495
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0007169052259996533
Simulation Start: 1956.7 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.000590154028031975
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0007025703089311719
Simulation Start: 1956.9 MB
Constrained in

In [ ]:
rho = -0.9
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1957.0 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0022261831909418106
Naive initialization. Warmup Length: 10; mean of MSE is: 0.00022255245130509138
Simulation Start: 1957.0 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.001300979987718165
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00013426030636765063
Simulation Start: 1957.0 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0006377717945724726
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0003869448264595121
Simulation Start: 1957.0 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0009173855069093406
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0006728458101861179
Simulation Start: 1957.0 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0007938941707834601
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0007175334030762315
Simulation Start: 1957.0 MB
Constrained

In [ ]:
from google.colab import runtime
runtime.unassign()